# Logistic Regression Lab

## Introduction

You are a junior data scientist working for a real estate investment firm that wants to classify neighborhoods into high-value and low-value housing markets. To help real estate developers and investors make strategic decisions, your team is building a logistic regression model to predict whether a house's value is above or below the median.

By analyzing features such as Median Income (MedInc), Average Rooms (AveRooms), and Average Occupancy (AveOccup), the company aims to identify key indicators of high-value neighborhoods. **The goal is to create a model that can accurately classify homes and help stakeholders prioritize investment opportunities.**

In real-world real estate analytics, classification models play a crucial role in risk assessment, property valuation, and investment planning.

This lab will help you:
- Convert a continuous target into a binary classification problem, simulating real-world decision-making.
- Scale and preprocess the data to improve model performance.
- Fit and evaluate a logistic regression model, interpreting coefficients and predictions.
- Assess model performance using a confusion matrix and accuracy score.
- Analyze Variance Inflation Factors (VIFs) to assess multicollinearity among predictor variables.

By completing this lab, you will gain practical experience in classification modeling, feature scaling, and multicollinearity analysis, which are essential skills in data-driven business applications.

### Step 0

Load the appropriate libraries and bring in the data. Note that we have to run a script to get the [California Housing dataset](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_california_housing.html) to match the version in `scikit-learn`. We cannot pull it directly from `scikit-learn` since CodeGrade cannot access the internet.

In [1]:
# CodeGrade step0

from sklearn.datasets import fetch_california_housing
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from scipy.stats import pearsonr
import os
import tarfile
import joblib # Import joblib directly
from sklearn.datasets._base import _pkl_filepath, get_data_home
import statsmodels.api as sm
import statsmodels.formula.api as smf
import seaborn as sns
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Process to get the data to match the version in scikit-learn:
archive_path = "cal_housing.tgz" # Change the path if it's not in the current directory.
data_home = get_data_home(data_home = None) # Change data_home if you are not using ~/scikit_learn_data.
if not os.path.exists(data_home):
    os.makedirs(data_home)
filepath = _pkl_filepath(data_home, "cal_housing.pkz")

with tarfile.open(mode = "r:gz", name = archive_path) as f:
    cal_housing = np.loadtxt(
                            f.extractfile("CaliforniaHousing/cal_housing.data"),
                            delimiter = ",")
    # Columns are not in the same order compared to the previous URL resource
    # on lib.stat.cmu.edu:
    columns_index = [8, 7, 2, 3, 4, 5, 6, 1, 0]
    cal_housing = cal_housing[:, columns_index]

    joblib.dump(cal_housing, filepath, compress = 6) # Now using the directly imported joblib.

# Load the dataset:
california = fetch_california_housing(as_frame = True)
data = california.data
data["MedianHouseValue"] = california.target

Print the basic information of the data using `.info()` and `.describe`.

In [2]:
# Display basic information:
print(data.info())
print(data.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   MedInc            20640 non-null  float64
 1   HouseAge          20640 non-null  float64
 2   AveRooms          20640 non-null  float64
 3   AveBedrms         20640 non-null  float64
 4   Population        20640 non-null  float64
 5   AveOccup          20640 non-null  float64
 6   Latitude          20640 non-null  float64
 7   Longitude         20640 non-null  float64
 8   MedianHouseValue  20640 non-null  float64
dtypes: float64(9)
memory usage: 1.4 MB
None
             MedInc      HouseAge      AveRooms     AveBedrms    Population  \
count  20640.000000  20640.000000  20640.000000  20640.000000  20640.000000   
mean       3.870671     28.639486      5.429000      1.096675   1425.476744   
std        1.899822     12.585558      2.474173      0.473911   1132.462122   
min        0.4

### Step 1

- Define `threshold` as the median of `MedianHouseValue`.
- Create a binary target variable called `HighValue` using the following code:
> `data["HighValue"] = (data["MedianHouseValue"] > threshold).astype(int)`

Return an array of the unique values in `HighValue`, called `unique_values`.

In [3]:
# CodeGrade step1

# Use the median `MedianHouseValue` as the threshold:
threshold = data["MedianHouseValue"].median()

# Create a binary target variable:
data["HighValue"] = (data["MedianHouseValue"] > threshold).astype(int)

# Find the unique values in the `HighValue` column:
unique_values = data["HighValue"].unique()

unique_values

array([1, 0])

### Step 2

- Select `MedInc`, `AveRooms`, and `AveOccup` as the variables of `X`, and let `y` be the variable `HighValue`.
- Set `seed` to 42.
- Split the data into `X_train`, `X_test`, `y_train`, and `y_test`, with a test size of 30% and a random state of 42.

Return the shapes of these four arrays in the same order as listed above.

In [4]:
# CodeGrade step2

# Define the predictors (X) and the response variable (y):
X = data[["MedInc", "AveRooms", "AveOccup"]]
y = data["HighValue"]

seed = 42
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, 
                                                    random_state = seed)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((14448, 3), (6192, 3), (14448,), (6192,))

### Step 3

- Use `scaler = StandardScaler()` to scale the data.
- Use `.fit_transform()` to fit and transform `X_train`, calling it `X_train_scaled`.
- Use `.transform()` to transform `X_test`, calling it `X_test_scaled`.

Return the shape of `X_test_scaled`.

In [5]:
# CodeGrade step3

# Scale the predictors:
scaler = StandardScaler()

# Transform the `X_train` and `X_test` predictors:
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), 
                              columns = X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns = X_test.columns)

X_test_scaled.shape

(6192, 3)

### Step 0

Run the code below without changes.

In [6]:
# CodeGrade step0

# Train the logistic regression model:
model = LogisticRegression()
model.fit(X_train_scaled, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default sol

### Step 4

Return the model's intercept.

In [7]:
# CodeGrade step4

model.intercept_

array([0.12755905])

### Step 5

Return the model's coefficients.

In [8]:
# CodeGrade step5

model.coef_

array([[ 2.33711986, -0.88891482, -2.55688063]])

### Step 6

Using the model:
- Predict the probabilities of `X_test_scaled`, calling it `y_pred_prob`.
- Predict the classes of `X_test_scaled`, calling it `y_pred_class`.

Return the first five elments of both of these arrays: `y_pred_prob` and `y_pred_class`.

In [9]:
# CodeGrade step6

# The predicted probabilities reflect the likelihood of each test observation
# belonging to the positive class (`HighValue` = 1/True). Predictions are
# converted to binary classes using a default threshold of 0.5, thereby
# capturing the model's decision boundary.

# Predict the probabilities of each house in `X_test_scaled` being `HighValue`
# = 1/True:
# `model.predict_proba()` actually gives two probabilities for every
# observation:
# Probability of 0    Probability of 1
# The two probabilities for each observation always add up to 1. The `[:, 1]` is
# telling Python: "Give me the probability corresponding to class 1, for every
# observation." That's why `y_pred_prob` contains just the `HighValue`
# probabilities.
y_pred_prob = model.predict_proba(X_test_scaled)[:, 1]

# Using these probabilities, and the threshold that prob ≥ 0.5 --> 1/True,
# predict the classes for each house in `X_test_scaled`:
y_pred_class = model.predict(X_test_scaled)

# Return the first five elements of both of these arrays:
y_pred_prob[0:5], y_pred_class[0:5]

(array([0.09349518, 0.21617122, 0.63030197, 0.88899024, 0.51316254]),
 array([0, 0, 1, 1, 1]))

### Step 7

Provide the confusion matrix of `y_test` and `y_pred_class`.

In [10]:
# CodeGrade step7

# A confusion matrix from `sklearn` is arranged as:
#           Predicted
#           0   1
# Actually
#        0  #   #
#        1  #   #
# This can also be represented as:
#                           Predicted
#                     0                     1
# Actually
#        0  # of True Negatives    # of False Positives
#        1  # of False Negatives   # of True Positives

confusion_matrix(y_test, y_pred_class)

array([[2478,  591],
       [ 810, 2313]])

### Step 8

Rounding to four decimal places, provide the accuracy score of `y_test` and `y_pred_class`.

In [11]:
# CodeGrade step8

# What is the proportion of all observations that were predicted correctly?:
round(accuracy_score(y_test, y_pred_class), 4)

0.7737

### Step 9

Rounding to three decimal places, provide the Variance Inflation Factors (VIFs) for each of the three columns of `X_train_scaled`.

In [12]:
# CodeGrade step9

# VIF (Variance Inflation Factor) measures multicollinearity, which is
# essentially how much one predictor can be explained by the other predictors
# in the model. A VIF of 1 means there is essentially no multicollinearity
# involving that predictor.

# Calculate the VIF for each predictor:
vif = pd.DataFrame()
vif["Variable"] = X_train.columns
vif["VIF"] = [variance_inflation_factor(X_train_scaled.values, i)
              for i in range(X_train_scaled.shape[1])]

# Round the VIFs to three decimal places:
round(vif, 3)

# The VIF values are all extremely close to 1:
# MedInc: 1.118
# AveRooms: 1.117
# AveOccup: 1.001
# So there's very little overlap/redundancy among these three predictors.
# That's good for interpreting the individual logistic regression coefficients.

,Variable,VIF
0,MedInc,1.118
1,AveRooms,1.117
2,AveOccup,1.001
